# README

# Limitation
- Survisorship bias: The initial dataset uses the current S&P 500 constituents (crawled from Wikipedia) and therefore contains survivorship bias.
- Globally defined valid data: the current data requires "["open", "low", "high", "close", "adj_close", "volume"]" to all exists, and in the cleaning step, removing all the rows lacking values of one of these fields. But this is problematic if the factors we are concerned with do not need all those values (e.g., momentum only needs "adjusted_close", so if "volume = NaN", it is fine, but our cleaning step will aggresivelly remove rows with "volume = NaN")
- The report in the cleaning step should be returned like a "dict", or "json", rather than simply "LOGGER"
- Need validation tests, like
""" python
assert panel.index.names == ["date", "ticker"]
assert panel.index.is_monotonic_increasing
assert panel["adj_close"].ge(0).all()
assert panel["volume"].ge(0).all()
assert panel.index.get_level_values("ticker").nunique() > 400
"""

In [1]:
from __future__ import annotations

import io
import logging
from pathlib import Path
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
LOGGER = logging.getLogger("phase0_pipeline")
CONFIG = {
    "universe_source_url": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    "start_date": "2015-01-01",
    "end_date": "2026-01-01",
    "momentum_lookback_days": 252,
    "momentum_skip_days": 21,
    "mean_reversion_lookback_days": 5,
    "volatility_lookback_days": 20,
    "forward_return_days": 5,
    "min_cross_section": 20,
    "cv_splits": 5,
    "min_train_days": 60,
    "ridge_alphas": np.logspace(-4, 4, 25),
    "top_quantile": 0.10,
    "bottom_quantile": 0.10,
    "risk_free_rate": 0.0,
    "annualization": 252,
    "data_dir": Path("data"),
    "ohlcv_path": Path("data/ohlcv.parquet"),
    "factor_panel_path": Path("data/factor_panel.parquet"),

}

# Data

In [3]:
def ensure_data_directory(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def save_parquet(df: pd.DataFrame, path: Path) -> None:
    ensure_data_directory(path)
    df.to_parquet(path)
    LOGGER.info("Saved DataFrame to %s", path)

class TickerCreator:
    def __init__(self, url:str):
        self._url = url

    def _read_html_tables(self) -> list[pd.DataFrame]:
        request = Request(
            self._url,
            headers = {
                "User-Agent": (
                    "Mozilla/5.0 (X11; Linux x86_64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/124.0 Safari/537.36"
                )
             },
        )
        with urlopen(request, timeout = 30) as response:
            html = response.read().decode("utf-8", errors="ignore")

        tables = pd.read_html(io.StringIO(html))
        if not tables:
            raise ValueError(f"No HTML tables were found at URL:{self._url}")
        return tables
    def get_sp500_tickers(self)-> list[str]:
        tables = self._read_html_tables()
        constituents = tables[0]
        if "Symbol" not in constituents.columns:
            raise ValueError("Expected a 'Symbol' column in the S&P 500 constituent table.")
        tickers = (
            constituents["Symbol"]
            .astype(str)
            .str.strip()
            .str.replace(".", "-", regex=False)
            .drop_duplicates()
            .tolist()
        )
        LOGGER.info("Loaded %d tickers from Wikipedia", len(tickers))
        return tickers

class DataCrawler:
    def __init__(self, start_date, end_date):
        self._start_date = start_date
        self._end_date = end_date

    def _standardize_downloaded_dataframe(self,raw: pd.DataFrame) -> pd.DataFrame:
        if raw.empty:
            raise ValueError("Received an empty DataFrame from yfinance.")

        if not isinstance(raw.columns, pd.MultiIndex):
            raw.columns = pd.MultiIndex.from_product([raw.columns, ["SINGLE_TICKER"]])

        first_level = [str(value).lower() for value in raw.columns.get_level_values(0)]
        second_level = [str(value).lower() for value in raw.columns.get_level_values(1)]
        known_fields = {"open", "close", "high", "low", "adj close", "volume"}

        if set(first_level).intersection(known_fields):
            ticker_level = 1
            field_level = 0
        elif set(second_level).intersection(known_fields):
            ticker_level = 0
            field_level = 1
        else:
            raise ValueError("Could not identify field and ticker levels in yfinance output")

        panel = (
            raw.stack(level = ticker_level, future_stack = True)
            .rename_axis(index = ["date", "ticker"])
            .sort_index()
        )

        panel.columns = [
            str(column).lower().replace(" ", "_").replace("/", "_")
            for column in panel.columns
        ]

        expected_columns = ["open", "high", "low", "close", "adj_close", "volume"]
        missing_columns = [column for column in expected_columns if column not in panel.columns]
        if missing_columns:
            raise ValueError(
                f"Missing expected columns from yfinance output: {missing_columns}"
            )

        return panel


    def down_load_ohlcv_data(self, tickers: list[str])->pd.DataFrame:
        if not tickers:
            raise ValueError("The ticker list is empty. At least one ticker is required.")

        LOGGER.info(
            "Downloading OHLCV for %d tickers from %s to %s",
            len(tickers),
            self._start_date,
            self._end_date
        )
        raw = yf.download(
            tickers=tickers,
            start=self._start_date,
            end=self._end_date,
            auto_adjust=False,
            progress=False,
            group_by="column",
            threads=True,
        )

        panel = self._standardize_downloaded_dataframe(raw)
        missing_mask = panel.isna().any(axis=1)
        if missing_mask.any():
            missing_counts = (
                missing_mask.groupby("ticker").sum().sort_values(ascending = False)
            )
            affected_tickers = missing_counts[missing_counts>0]
            for ticker, count in affected_tickers.items():
                LOGGER.warning(
                    "Ticker %s has %d ticker-date rows with missing OHLCV fields; those rows will be dropped.",
                    ticker,
                    int(count),
                )

        missing_ratio = (
            panel["adj_close"].isna().groupby(level="ticker").mean().sort_values(ascending=False)
        )
        warn_tickers = missing_ratio[missing_ratio > 0.05]
        for ticker, ratio in warn_tickers.items():
            LOGGER.warning(
                "Ticker %s has %.2f%% missing adjusted-close observations in the raw download.",
                ticker,
                ratio * 100.0,
            )

        panel = panel.dropna(subset = ["open", "low", "high", "close", "adj_close", "volume"])
        if panel.empty:
            raise ValueError("All downloaded OHLCV rows were dropped after missing-data cleaning.")

        LOGGER.info(
            f"Cleaned OHLCV panel shape: %s with %d unique tickers",
            panel.shape,
            panel.index.get_level_values("ticker").nunique(),
        )
        return panel


In [4]:
ticker_creator = TickerCreator(CONFIG['universe_source_url'])
data_crawler = DataCrawler(CONFIG["start_date"], CONFIG["end_date"])

tickers = ticker_creator.get_sp500_tickers()
ohlcv = data_crawler.down_load_ohlcv_data(tickers)

2026-08-10 21:29:47,886 | INFO | Loaded 503 tickers from Wikipedia
2026-08-10 21:29:47,888 | INFO | Downloading OHLCV for 503 tickers from 2015-01-01 to 2026-01-01
2026-08-10 21:29:47,888 | INFO | Downloading OHLCV for 503 tickers from 2015-01-01 to 2026-01-01
2026-08-10 21:30:49,173 | ERROR | 
2 Failed downloads:
2026-08-10 21:30:49,175 | ERROR | ['FDXF', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2026-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1767243600")')
2026-08-10 21:30:53,804 | WARNING | Ticker HONA has 2766 ticker-date rows with missing OHLCV fields; those rows will be dropped.
2026-08-10 21:30:53,805 | WARNING | Ticker FDXF has 2766 ticker-date rows with missing OHLCV fields; those rows will be dropped.
2026-08-10 21:30:53,806 | WARNING | Ticker Q has 2720 ticker-date rows with missing OHLCV fields; those rows will be dropped.
2026-08-10 21:30:53,807 | WARNING | Ticker SNDK has 2544 ticker-dat

In [5]:
save_parquet(ohlcv, CONFIG["ohlcv_path"])
display(ohlcv.head())
display(
    ohlcv.groupby(level="ticker").size().sort_values(ascending=False).head().rename("rows_per_ticker")
)

2026-08-10 21:30:56,366 | INFO | Saved DataFrame to learn/data/ohlcv.parquet


adj_close      close       high        low       open  \
date       ticker                                                          
2015-01-02 A       36.899391  40.560001  41.310001  40.369999  41.180000   
           AAPL    24.192600  27.332500  27.860001  26.837500  27.847500   
           ABBV    41.119225  65.889999  66.400002  65.440002  65.440002   
           ABT     35.754211  44.900002  45.450001  44.639999  45.250000   
           ACGL    18.539352  19.496668  19.860001  19.426666  19.733334   

                        volume  
date       ticker               
2015-01-02 A         1529200.0  
           AAPL    212818400.0  
           ABBV      5086100.0  
           ABT       3216600.0  
           ACGL      1101600.0

ticker
ZTS     2766
A       2766
AAPL    2766
ABBV    2766
WFC     2766
Name: rows_per_ticker, dtype: int64